# 03 — Nested outer-CV model development

Primary performance evidence comes from five outer stratified folds over the full labelled population. Logistic regularization is selected by three inner folds; LightGBM candidate and iteration selection use an inner split only. The previously used 80/20 split is reconstructed solely as a legacy holdout for notebook 04 comparability.

In [1]:
from pathlib import Path
import json
import joblib
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.modeling import run_development

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
MODELS = ROOT / "models"
MODELS.mkdir(parents=True, exist_ok=True)

features = pd.read_pickle(PROCESSED / "borrower_features.pkl")
development = run_development(features)

In [2]:
pd.DataFrame({"SK_ID_CURR": development.split_ids["development"]}).to_csv(PROCESSED / "development_ids.csv", index=False)
pd.DataFrame({"SK_ID_CURR": development.split_ids["legacy_holdout"]}).to_csv(PROCESSED / "legacy_holdout_ids.csv", index=False)
development.outer_cv_summary.to_csv(TABLES / "outer_cv_model_comparison.csv", index=False)
development.logistic_stability.to_csv(TABLES / "logistic_coefficients.csv", index=False)
joblib.dump(development, MODELS / "development_result.joblib")

selected_auc = development.outer_cv_summary[["model", "roc_auc_mean", "roc_auc_std"]]
log_auc = float(selected_auc.loc[selected_auc["model"].eq("Logistic"), "roc_auc_mean"].iloc[0])
lgb_auc = float(selected_auc.loc[selected_auc["model"].eq("LightGBM"), "roc_auc_mean"].iloc[0])
primary = development.primary_model
lock = {
    "primary_validation": "5-fold outer StratifiedKFold over all labelled rows, shuffle=True, random_state=42",
    "legacy_holdout": "previously used 80/20 split retained for comparability; not an independent test",
    "inner_logistic": "3-fold StratifiedKFold inside each outer train",
    "inner_lightgbm": "stratified 80/20 split inside each outer train for candidate and iteration selection",
    "logistic_c": development.selected_c,
    "lightgbm_params": development.selected_lgbm_params,
    "lightgbm_iterations": development.selected_lgbm_iterations,
    "calibration": development.calibration,
    "capacity_levels": [0.05, 0.10, 0.20],
    "primary_model": primary,
    "primary_rule": "LightGBM only if outer CV AUC exceeds Logistic by at least 0.005; otherwise prefer transparency",
}
(MODELS / "design_lock.json").write_text(json.dumps(lock, indent=2), encoding="utf-8")
display(selected_auc)
display(lock)

,model,roc_auc_mean,roc_auc_std
0,Application-only Logistic,0.657943,0.003843
1,Logistic,0.723163,0.001218
2,LightGBM,0.753450,0.002978


{'primary_validation': '5-fold outer StratifiedKFold over all labelled rows, shuffle=True, random_state=42',
 'legacy_holdout': 'previously used 80/20 split retained for comparability; not an independent test',
 'inner_logistic': '3-fold StratifiedKFold inside each outer train',
 'inner_lightgbm': 'stratified 80/20 split inside each outer train for candidate and iteration selection',
 'logistic_c': 0.1,
 'lightgbm_params': {'num_leaves': 15,
  'max_depth': 5,
  'min_child_samples': 100,
  'learning_rate': 0.04,
  'colsample_bytree': 0.8,
  'subsample': 0.8,
  'reg_alpha': 0.0,
  'reg_lambda': 3.0},
 'lightgbm_iterations': 1114,
 'calibration': {'Application-only Logistic': {'applied': False,
   'raw_brier': 0.07226934126618353,
   'slope': 0.9934058025410256,
   'intercept': -0.01570042195501144},
  'Logistic': {'applied': False,
   'raw_brier': 0.06985212142165739,
   'slope': 0.9912871653873263,
   'intercept': -0.020233095847441124},
  'LightGBM': {'applied': False,
   'raw_brier': 

In [3]:
app_auc = float(selected_auc.loc[selected_auc["model"].eq("Application-only Logistic"), "roc_auc_mean"].iloc[0])
print(f"Incremental-history CV AUC: {log_auc - app_auc:+.4f} (full Logistic versus application-only Logistic)")
assert len(development.split_ids["development"]) + len(development.split_ids["legacy_holdout"]) == len(features)
assert set(development.split_ids["development"]).isdisjoint(development.split_ids["legacy_holdout"])
assert not development.selection_audit["outer_validation_used_for_early_stopping"].any()
print("Primary evidence locked from nested outer CV. The legacy holdout is reserved for descriptive comparability only.")

Incremental-history CV AUC: +0.0652 (full Logistic versus application-only Logistic)
Primary evidence locked from nested outer CV. The legacy holdout is reserved for descriptive comparability only.
